# AI Agent Loop Demo


This notebook demonstrates the core mechanism behind an AI agent: a model that can decide to call a tool, receive the result, and continue reasoning toward a goal — all in a loop.

---

This notebook contains starter code. Please fill in details as needed.
---

## Setup - Gemini API key

How to get the key (takes about two minutes):

* Go to [aistudio.google.com](https://aistudio.google.com/prompts/new_chat)
* Sign in with your Google account
* Click "Get API key" in the left sidebar
* Click "Create API key" — choose "Create API key in new project" if prompted
Copy the key that appears (it starts with AIza...)

> The free tier allows up to 10 requests per minute and 250 requests per day, which is enough to work through this notebook.

If you see occasional slow-downs when you run several questions back to back — that is normal, and the code handles it automatically.

## References
* [Google agent loop](https://github.com/xiaosuhu/midas-ai-in-research/blob/v1.0-dev/docs/notebooks/agent_loop_demo.ipynb$0)
* https://ai.google.dev/gemini-api/docs/thinking#signatures
* https://ai.google.dev/gemini-api/docs/text-generation$0
* [Developer guide](https://github.com/xiaosuhu/midas-ai-in-research/blob/v1.0-dev/docs/notebooks/agent_loop_demo.ipynb$0)



In [1]:
import os

# NEVER share API keys directly. Save them in the environment.
try:
    from google.colab import userdata
    os.environ['GOOGLE_API_KEY'] = userdata.get('GOOGLE_API_KEY')
    print('API key loaded from Colab Secrets.')

# Fall back to manual entry if Secrets are not set up.
# REMOVE API keys before pushing code to Github. This fallback helps.
except Exception:
    import getpass
    key = getpass.getpass('Paste your Gemini API key and press Enter: ')
    os.environ['GOOGLE_API_KEY'] = key
    print('API key set.')

Paste your Gemini API key and press Enter: ··········
API key set.


In [1]:
# Quiet ensures your UI is not full of logs.
!pip install google-generativeai --quiet
print("Done.")

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-cloud-datastore 2.27.0 requires protobuf<8.0.0,>=6.33.5, but you have protobuf 5.29.6 which is incompatible.
grpc-google-iam-v1 0.14.5 requires protobuf<8.0.0,>=6.33.5, but you have protobuf 5.29.6 which is incompatible.
google-cloud-dataproc 5.32.0 requires protobuf<8.0.0,>=6.33.5, but you have protobuf 5.29.6 which is incompatible.
tensorboard 2.21.0 requires protobuf<8.0.0,>=6.31.1, but you have protobuf 5.29.6 which is incompatible.
google-cloud-language 2.22.0 requires protobuf<8.0.0,>=6.33.5, but you have protobuf 5.29.6 which is incompatible.
tensorflow 2.21.0 requires protobuf<8.0.0,>=6.31.1, but you have protobuf 5.29.6 which is incompatible.
google-cloud-storage-control 1.16.0 requires protobuf<8.0.0,>=6.33.5, but you have protobuf 5.29.6 which is incompatible.
google-cloud-bigquery-connection 1.2

In [5]:
# @title User input needed.

# TODO: Fill in documents in the format mentioned in the first example.
DOCUMENTS = {

        "ai_basics": (
        "Artificial intelligence is a branch of computer science that develops "
        "systems capable of performing tasks that normally require human intelligence. "
        "These tasks include learning, reasoning, problem-solving, and understanding language. "
        "AI systems can process large amounts of data to identify patterns and make predictions. "
        "Examples of AI include virtual assistants, recommendation systems, and chatbots."
    ),

    "machine_learning": (
        "Machine learning is a subset of artificial intelligence that enables computers "
        "to learn patterns from data without being explicitly programmed for every task. "
        "Supervised learning uses labelled data to train a model. "
        "Unsupervised learning discovers patterns in data without predefined labels. "
        "Deep learning is a type of machine learning that uses neural networks with multiple layers."
    ),

    "ai_applications": (
        "Artificial intelligence is used in healthcare to help analyze medical images "
        "and support clinical decision-making. "
        "In education, AI tools can provide personalized learning experiences for students. "
        "In agriculture, AI can help monitor crops and detect plant diseases. "
        "In transportation, AI supports route planning, traffic prediction, and driver assistance."
    ),

    "ai_ethics": (
        "AI ethics concerns the responsible design and use of artificial intelligence systems. "
        "AI developers should consider fairness and avoid unjust discrimination. "
        "Personal information should be protected when AI systems collect or process data. "
        "AI decisions should be explainable when they significantly affect people's lives. "
        "Human oversight is important for high-impact decisions made with AI assistance."
    )
}

In [6]:
# @title Define a tool
def search_documents(query: str) -> str:
    """Search the document collection for paragraphs relevant to the query."""
    query_lower = query.lower()
    results = []

    for doc_name, content in DOCUMENTS.items():
        words = set(query_lower.split())
        match_count = sum(1 for word in words if word in content.lower())
        if match_count > 0:
            results.append((match_count, doc_name, content))

    results.sort(reverse=True)

    if not results:
        return "No relevant documents found for that query."

    # Return the top two matches
    output = []
    for _, name, content in results[:2]:
        output.append(f"[Source: {name}]\n{content}")
    return "\n\n".join(output)


# Sanity check for regex match.
print(search_documents("minimum of 4"))


[Source: machine_learning]
Machine learning is a subset of artificial intelligence that enables computers to learn patterns from data without being explicitly programmed for every task. Supervised learning uses labelled data to train a model. Unsupervised learning discovers patterns in data without predefined labels. Deep learning is a type of machine learning that uses neural networks with multiple layers.

[Source: ai_ethics]
AI ethics concerns the responsible design and use of artificial intelligence systems. AI developers should consider fairness and avoid unjust discrimination. Personal information should be protected when AI systems collect or process data. AI decisions should be explainable when they significantly affect people's lives. Human oversight is important for high-impact decisions made with AI assistance.


In [31]:
# @title Ask gemini to search for something
import google.generativeai as genai
import time
import os
from google.colab import userdata

genai.configure(api_key=userdata.get("API"))

model = genai.GenerativeModel(
    # This will break.
    # TODO: Find the right model to call in this cell from developer docs.
    model_name="gemini-3.5-flash-lite",
    tools=[search_documents],
    system_instruction=(
                """
        You are an AI assistant answering questions about Artificial Intelligence.
        Use the search_documents tool when you need information from the documents.
        Only answer based on what you find in the documents.
        If the documents do not contain the answer, say so clearly.
        """
    )
)

print("Model ready:", model.model_name)

Model ready: models/gemini-3.5-flash-lite


In [34]:
import time
from google import genai
from google.genai import types
from google.colab import userdata


client = genai.Client(api_key=userdata.get("API"))  # reads GEMINI_API_KEY / GOOGLE_API_KEY from env

search_tool = types.Tool(function_declarations=[
    types.FunctionDeclaration(
        name="search_documents",
        description="Search the documents to find most the relevant paragraph.",
        parameters={
            "type": "object",
            "properties": {"query": {"type": "string"}},
            "required": ["query"],
        },
    )
])

config = types.GenerateContentConfig(
    tools=[search_tool],
    automatic_function_calling=types.AutomaticFunctionCallingConfig(disable=True),
)


def send_with_retry(contents, max_retries=3):
    for attempt in range(max_retries):
        try:
            return client.models.generate_content(
                model="gemini-3.5-flash-lite",
                contents=contents, config=config
            )
        except Exception as e:
            if "429" in str(e) and attempt < max_retries - 1:
                wait = 20 * (attempt + 1)
                print(f"  Rate limit hit, waiting {wait}s...")
                time.sleep(wait)
            else:
                raise


def run_agent(question: str, max_steps: int = 8) -> str:
    print(f"Question: {question}\n" + "-" * 50)
    contents = [types.Content(role="user", parts=[types.Part(text=question)])]

    for step in range(1, max_steps + 1):
        response = send_with_retry(contents)
        model_content = response.candidates[0].content

        # Append the model's turn UNMODIFIED, so thought signatures are preserved
        contents.append(model_content)

        calls = [p.function_call for p in model_content.parts if p.function_call]
        if not calls:
            print(f"\nFinal answer:\n{response.text}")
            return response.text

        # Run every requested tool call, reply with all results in ONE user turn
        result_parts = []
        for call in calls:
            args = dict(call.args)
            print(f"Step {step}: model calls {call.name}({args})")
            result = search_documents(**args)
            print(f"  Result preview: {str(result)[:100]}...")
            result_parts.append(
                types.Part.from_function_response(
                    name=call.name, response={"result": result}
                )
            )
        contents.append(types.Content(role="user", parts=result_parts))

    return "(Stopped: hit max_steps)"


_ = run_agent("What is artificial intelligence?")
print("Testing Gemini connection...")

response = client.models.generate_content(
    model="gemini-3.5-flash-lite",
    contents="Reply with just: Hello"
)

print(response.text)

Question: What is artificial intelligence?
--------------------------------------------------
Step 1: model calls search_documents({'query': 'artificial intelligence definition'})
  Result preview: [Source: machine_learning]
Machine learning is a subset of artificial intelligence that enables comp...

Final answer:
**Artificial Intelligence (AI)** refers to the development of computer systems and software capable of performing tasks that typically require human intelligence. These tasks include learning, reasoning, problem-solving, perception, language understanding, and decision-making. 

Key aspects of artificial intelligence include:

* **Machine Learning:** A prominent subset of AI that allows computers to learn patterns from data and improve their performance over time without being explicitly programmed for every individual task.
* **Deep Learning:** An advanced branch of machine learning utilizing multi-layered neural networks to analyze complex data patterns, such as images, au

In [29]:
from google import genai
from google.colab import userdata

client = genai.Client(api_key=userdata.get("API"))

try:
    for m in client.models.list():
        if "generateContent" in (m.supported_actions or []):
            print(m.name)
except Exception as e:
    print(type(e).__name__, e)

models/gemini-2.5-flash
models/gemini-2.5-pro
models/gemini-2.5-flash-preview-tts
models/gemini-2.5-pro-preview-tts
models/gemma-4-26b-a4b-it
models/gemma-4-31b-it
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-pro-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-3-flash-preview
models/gemini-3.1-pro-preview
models/gemini-3.1-pro-preview-customtools
models/gemini-3.1-flash-lite-preview
models/gemini-3.1-flash-lite
models/gemini-3-pro-image-preview
models/gemini-3-pro-image
models/nano-banana-pro-preview
models/gemini-3.1-flash-image-preview
models/gemini-3.1-flash-image
models/gemini-3.1-flash-lite-image
models/gemini-nano-banana-2.1
models/gemini-3.5-flash
models/gemini-3.5-flash-lite
models/gemini-omni-flash-preview
models/gemini-omni-1.1-flash
models/gemini-3.5-transcribe
models/gemini-3.6-flash
models/gemini-3.7-flash
models/gemini-3.8-flash
models/gemini-4-argon
models/lyria-3-clip-preview
models/lyria-3-pro-preview
models/l

In [30]:
from google.genai import types

print("Testing Gemini...", flush=True)

try:
    response = client.models.generate_content(
        model="gemini-3.5-flash-lite",
        contents="Reply with only the word Hello.",
        config=types.GenerateContentConfig(
            max_output_tokens=10,
            http_options=types.HttpOptions(timeout=30000)
        )
    )
    print("Success:", response.text)

except Exception as e:
    print("Error:", type(e).__name__, str(e))

Testing Gemini...
Success: Hello.


In [36]:
from google.genai import types

response = client.models.generate_content(
    model="gemini-3.5-flash-lite",
    contents="Say Hello.",
    config=types.GenerateContentConfig(
        max_output_tokens=10,
        http_options=types.HttpOptions(timeout=60000)
    )
)

print(response.text)

Hello! How can I help


In [40]:
# @title Ask follow up questions

# TODO: Enter questions based on documents for the agent to answer.
questions = [
        "What is the difference between artificial intelligence and machine learning?",
    # This question is not answered by the database.
        "How many ghosts eat carrots everyday?",
    # This question is part of the general knowledge of the model.
    "What are the key ethical concerns when developing AI systems?",
     "What are the main applications of artificial intelligence in education and healthcare?",
        "Why are fairness, privacy, and human oversight important in AI ethics?",


]

for i, q in enumerate(questions):
    if i > 0:
        time.sleep(15)  # stay within the free-tier rate limit.
    print("\n" + "=" * 60)
    run_agent(q)



Question: What is the difference between artificial intelligence and machine learning?
--------------------------------------------------
Step 1: model calls search_documents({'query': 'artificial intelligence machine learning difference'})
  Result preview: [Source: machine_learning]
Machine learning is a subset of artificial intelligence that enables comp...

Final answer:
The primary difference between artificial intelligence (AI) and machine learning (ML) comes down to scope: **AI is the broad umbrella concept, while ML is a specific subset of that umbrella.**

Here is a closer look at each:

*   **Artificial Intelligence (AI):** AI is a broad branch of computer science focused on creating systems capable of performing tasks that typically require human intelligence. This includes reasoning, problem-solving, perception, language understanding, and decision-making. AI can be achieved through hard-coded rules and logic (such as a traditional chess-playing program) or through data-dr

In [41]:
# This cell will check if you have added atleast 2 more questions.
assert(len(questions) >= 4)
# This will check if you added a question.
assert("Add a question here." not in questions)